In [1]:
from random import seed, choice, shuffle
import numpy as np
import logging
import heapq
from collections.abc import Iterator
from collections.abc import Callable

logging.getLogger().setLevel(logging.INFO)

# Classes

## Class for the priority Queue

In [2]:
class UPQ:
    _data: list[tuple[object, object]]
    _priority: dict

    def __init__(self, data=tuple()) -> None:
        """Create a new UPQ, possibly initialized with data."""
        self._data = list()
        self._priority = dict()
        for i in data:
            self.push_update(*i)

    def __getitem__(self, item) -> object:
        """Return the priority of an item."""
        return self._priority[item]

    def __len__(self) -> int:
        """Return the number of items in the queue."""
        return len(self._priority)

    def __bool__(self) -> bool:
        """Return True if the queue is not empty."""
        return bool(self._priority)

    def __delitem__(self, item) -> None:
        """Delete an item from the queue."""
        del self._priority[item]

    def __contains__(self, item) -> bool:
        """Check if item is in the queue."""
        return item in self._priority

    def __iter__(self) -> Iterator[tuple[object, object]]:
        """Iterate in priority order without modifying the queue."""
        return iter(sorted(self._priority.items(), key=lambda k: k[1]))

    def get(self, item, default=None) -> object:
        """Return the priority of an item, or default if not present."""
        return self._priority.get(item, default)

    def push_update(self, item, priority) -> None:
        """Push a new item with the given priority or update its priority."""
        self._priority[item] = priority
        heapq.heappush(self._data, (priority, item))

    def pop(self) -> tuple[object, object]:
        """Pop the item with the lowest priority. Fails if empty."""
        p, i = heapq.heappop(self._data)
        while p != self._priority.get(i):
            p, i = heapq.heappop(self._data)
        del self._priority[i]
        return (i, p)

    def pop_safe(self, default=None) -> tuple[object, object] | object:
        """Pop the item with the lowest priority. Return `default` if the queue is empty."""
        while self._data:
            p, i = heapq.heappop(self._data)
            if p == self._priority.get(i):
                del self._priority[i]
                return (i, p)
        else:
            return default

## Class for the State

In [3]:
class State:
    def __init__(self, data: np.ndarray, shapes: np.ndarray):
        self._data = data.copy()
        self._data.flags.writeable = False
        self._shapes = shapes
        self._key = bytes(shapes[self._data + 2])  # board seen as shapes, not labels

    def __hash__(self):
        return hash(self._key)

    def __eq__(self, other):
        return self._key == other._key

    def __lt__(self, other):
        return self._key < other._key

    def __str__(self):
        return str(self._data)

    def __repr__(self):
        return repr(self._data)

    @property
    def data(self):
        return self._data

    @property
    def shapes(self):
        return self._shapes

    def copy_data(self):
        return self._data.copy()


## Class for each Piece of the Puzzle

In [4]:
class Piece:
    def __init__(self, label, occupied_squares):
        self._label = label
        self._occupied_squares = occupied_squares

    def label(self):
        return self._label

    def moves(self):
        return [np.array(_) for _ in [(-1, 0), (0, +1), (+1, 0), (0, -1)]]

    def find_empty_space(self, board: np.ndarray):
        t = np.where(board == -1)
        return np.array([t[0], t[1]])

    def is_valid(self, board: np.ndarray, action):
        for occupied_square in self._occupied_squares:
            move = occupied_square + action
            # for free_space in self.find_empty_space(board):
            #     check_borders = all(0 <= (free_space + action)[i] < board.shape[i] for i in [0, 1])
            #     if not check_borders:
            #         return False
            if move[0] < 0 or move[0] >= board.shape[0] or move[1] < 0 or move[1] >= board.shape[1]:
                return False
            if not board[move[0], move[1]] in [-1, self.label()]:
                return False
        return True

    def possible_actions(self, state: State):
        moves = self.moves()
        return (m for m in moves if self.is_valid(state._data, m))

    def piece_moving(self, action):
        new_squares = []
        for occupied_square in self._occupied_squares:
            move = occupied_square + action
            new_squares.append(move)
        return Piece(self._label, new_squares)


# Functions

## Helpers

In [5]:
def pieces_from_state(state: State) -> list[Piece]:
    board = state.data
    return [Piece(label, list(np.argwhere(board == label))) for label in np.unique(board[board >= 0])]


def double_loop(state: State):
    for piece in pieces_from_state(state):
        for a in piece.possible_actions(state):
            yield (piece, a)

def result(state: State, action) -> State:
    piece, a = action
    board = state.copy_data()
    for r, c in piece._occupied_squares:
        board[r, c] = -1
    for r, c in piece.piece_moving(a)._occupied_squares:
        board[r, c] = piece.label()
    return State(board, state.shapes)

EMPTY = -1  # free cell
WALL = -2   # fixed cell that never moves (for non-rectangular boards)

def shape_codes(board: np.ndarray, target: int) -> np.ndarray:
    """Lookup table label -> shape code; pieces with the same shape share a code.
    Indexed by label + 2, so WALL -> index 0 and EMPTY -> index 1."""
    codes = np.zeros(board.max() + 3, dtype=np.uint8)
    codes[EMPTY + 2] = 1
    shapes = {}
    for label in np.unique(board[board >= 0]):
        cells = np.argwhere(board == label)
        shape = tuple(map(tuple, cells - cells.min(axis=0)))
        codes[label + 2] = shapes.setdefault(shape, len(shapes) + 2)
    codes[target + 2] = len(shapes) + 2
    return codes



## Goal Test

In [6]:
seed(123)
TARGET = 0                              # label of the 2x2 piece

def goal_test(state: State, goal_rows, goal_cols) -> bool:
    return bool((state.data[goal_rows, goal_cols] == TARGET).all())


## Search Algorithm

In [7]:
def search(initial_state: State, goal_test: Callable, parent_state: dict,
           state_cost: dict, priority_function: Callable, unit_cost: Callable, goal_rows, goal_cols):
    frontier = UPQ()
    parent_state.clear()
    state_cost.clear()

    state = initial_state
    parent_state[state] = None
    state_cost[state] = 0
    while state is not None and not goal_test(state, goal_rows, goal_cols):
        for a in double_loop(state):
            new_state = result(state, a)
            cost = unit_cost(a)
            if new_state not in state_cost and new_state not in frontier:
                parent_state[new_state] = state
                state_cost[new_state] = state_cost[state] + cost
                frontier.push_update(new_state, priority_function(new_state))
                logging.debug(f"Added new node to frontier (cost={state_cost[new_state]})")
            elif new_state in frontier and state_cost[new_state] > state_cost[state] + cost:
                old_cost = state_cost[new_state]
                parent_state[new_state] = state
                state_cost[new_state] = state_cost[state] + cost
                frontier.push_update(new_state, priority_function(new_state))
                logging.debug(f"Updated node cost in frontier: {old_cost} -> {state_cost[new_state]}")

        if frontier:
            state, _ = frontier.pop()
        else:
            state = None
    path = list()
    s = state
    while s:
        path.append(s.copy_data())
        s = parent_state[s]

    logging.info(f"Found a solution in {len(path):,} steps; visited {len(state_cost):,} states")
    return list(reversed(path)), len(state_cost)

# Scramble Initialization

In [8]:
results_scramble = {"Level 1": {"Goal": None, "Initial State": None,
                                "Algorithms": {"Breadth-First": {"Steps": None, "States": None},
                                               "Depth-First": {"Steps": None, "States": None},
                                               "Greedy Best-First": {"Steps": None, "States": None},
                                               "A*": {"Steps": None, "States": None},
                                               "A* (Manhattan)": {"Steps": None, "States": None}
                                               }
                                },
                    "Level 2": {"Goal": None, "Initial State": None,
                                "Algorithms": {"Breadth-First": {"Steps": None, "States": None},
                                               "Depth-First": {"Steps": None, "States": None},
                                               "Greedy Best-First": {"Steps": None, "States": None},
                                               "A*": {"Steps": None, "States": None},
                                               "A* (Manhattan)": {"Steps": None, "States": None}
                                               }
                                },
                    "Level 3": {"Goal": None, "Initial State": None,
                                "Algorithms": {"Breadth-First": {"Steps": None, "States": None},
                                               "Depth-First": {"Steps": None, "States": None},
                                               "Greedy Best-First": {"Steps": None, "States": None},
                                               "A*": {"Steps": None, "States": None},
                                               "A* (Manhattan)": {"Steps": None, "States": None}
                                               }
                                },
                    "Level 4": {"Goal": None, "Initial State": None,
                                "Algorithms": {"Breadth-First": {"Steps": None, "States": None},
                                               "Depth-First": {"Steps": None, "States": None},
                                               "Greedy Best-First": {"Steps": None, "States": None},
                                               "A*": {"Steps": None, "States": None},
                                               "A* (Manhattan)": {"Steps": None, "States": None}
                                               }
                                },
                    "Level 5": {"Goal": None, "Initial State": None,
                                "Algorithms": {"Breadth-First": {"Steps": None, "States": None},
                                               "Depth-First": {"Steps": None, "States": None},
                                               "Greedy Best-First": {"Steps": None, "States": None},
                                               "A*": {"Steps": None, "States": None},
                                               "A* (Manhattan)": {"Steps": None, "States": None}
                                               }
                                },
                    "Level 6": {"Goal": None, "Initial State": None,
                                "Algorithms": {"Breadth-First": {"Steps": None, "States": None},
                                               "Depth-First": {"Steps": None, "States": None},
                                               "Greedy Best-First": {"Steps": None, "States": None},
                                               "A*": {"Steps": None, "States": None},
                                               "A* (Manhattan)": {"Steps": None, "States": None}
                                               }
                                }
                    }

## Level 1 - 3x3 board with 1x1 target, 6 1x1 pieces, 2 empty cells

In [9]:
level1_dict = results_scramble["Level 1"]
start = np.array([[-1, -1, 1], [2, 3, 4], [5, 0, 6]])
GOAL = State(start, shape_codes(start, TARGET))
GOAL_ROWS, GOAL_COLS = slice(2, 3), slice(1, 2)   # rows 2, columns 1
level1_dict["Goal"] = GOAL
logging.info(f"Goal: \n{GOAL}")

INFO:root:Goal: 
[[-1 -1  1]
 [ 2  3  4]
 [ 5  0  6]]


In [10]:
INITIAL_STATE = GOAL
for r in range(100_000):
    INITIAL_STATE = result(INITIAL_STATE, choice(list(double_loop(INITIAL_STATE))))

level1_dict["Initial State"] = INITIAL_STATE
logging.info(f"Initial State: \n{INITIAL_STATE}")

INFO:root:Initial State: 
[[ 4  1  3]
 [ 0  6 -1]
 [ 2  5 -1]]


In [11]:
level1_dict_algo = level1_dict["Algorithms"]

### Breadth - First

In [12]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)
level1_dict_algo["Breadth-First"]["Steps"] = len(final[0])
level1_dict_algo["Breadth-First"]["States"] = final[1]

INFO:root:Found a solution in 5 steps; visited 66 states


### Depth-First

In [13]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: -len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)
level1_dict_algo["Depth-First"]["Steps"] = len(final[0])
level1_dict_algo["Depth-First"]["States"] = final[1]

INFO:root:Found a solution in 75 steps; visited 202 states


### Greedy Best-First

In [14]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: -len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)
level1_dict_algo["Greedy Best-First"]["Steps"] = len(final[0])
level1_dict_algo["Greedy Best-First"]["States"] = final[1]

INFO:root:Found a solution in 75 steps; visited 202 states


### A*

In [15]:
parent_state = dict()
state_cost = dict()


def h(state):
    return np.sum((state._data != GOAL._data) & (state._data > 0))

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: state_cost[s] + h(s),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)

level1_dict_algo["A*"]["Steps"] = len(final[0])
level1_dict_algo["A*"]["States"] = final[1]

INFO:root:Found a solution in 7 steps; visited 73 states


### A* (Manhattan Distance)

In [16]:
parent_state = dict()
state_cost = dict()

def h(state: State) -> int:
    r, c = np.argwhere(state.data == TARGET).min(axis=0)   # top-left cell of the 2x2
    return abs(r - GOAL_ROWS.start) + abs(c - GOAL_COLS.start)

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: state_cost[s] + h(s),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)

level1_dict_algo["A* (Manhattan)"]["Steps"] = len(final[0])
level1_dict_algo["A* (Manhattan)"]["States"] = final[1]

INFO:root:Found a solution in 5 steps; visited 30 states


## Level 2 - 4x4 board with 2x2 target, 10 1x1, 2 empty cells

In [17]:
level2_dict = results_scramble["Level 2"]
start = np.array([[-1, -1, 1, 2], [3, 4, 5, 6], [7, 0, 0, 8], [9, 0, 0, 10]])
GOAL = State(start, shape_codes(start, TARGET))
GOAL_ROWS, GOAL_COLS = slice(2, 4), slice(1, 3)   # rows 2-3, columns 1-2
level2_dict["Goal"] = GOAL
logging.info(f"Goal: \n{GOAL}")

INFO:root:Goal: 
[[-1 -1  1  2]
 [ 3  4  5  6]
 [ 7  0  0  8]
 [ 9  0  0 10]]


In [18]:
INITIAL_STATE = GOAL
for r in range(100_000):
    INITIAL_STATE = result(INITIAL_STATE, choice(list(double_loop(INITIAL_STATE))))

level2_dict["Initial State"] = INITIAL_STATE
logging.info(f"Initial State: \n{INITIAL_STATE}")

INFO:root:Initial State: 
[[ 5  0  0  9]
 [10  0  0  4]
 [ 3 -1 -1  2]
 [ 1  7  6  8]]


In [19]:
level2_dict_algo = level2_dict["Algorithms"]

### Breadth - First

In [20]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows=GOAL_ROWS,
    goal_cols=GOAL_COLS,
)
level2_dict_algo["Breadth-First"]["Steps"] = len(final[0])
level2_dict_algo["Breadth-First"]["States"] = final[1]

INFO:root:Found a solution in 13 steps; visited 337 states


### Depth-First

In [21]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: -len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)
level2_dict_algo["Depth-First"]["Steps"] = len(final[0])
level2_dict_algo["Depth-First"]["States"] = final[1]

INFO:root:Found a solution in 99 steps; visited 259 states


### Greedy Best-First

In [22]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: -len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)
level2_dict_algo["Greedy Best-First"]["Steps"] = len(final[0])
level2_dict_algo["Greedy Best-First"]["States"] = final[1]

INFO:root:Found a solution in 99 steps; visited 259 states


### A*

In [23]:
parent_state = dict()
state_cost = dict()


def h(state):
    return np.sum((state._data != GOAL._data) & (state._data > 0))

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: state_cost[s] + h(s),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)

level2_dict_algo["A*"]["Steps"] = len(final[0])
level2_dict_algo["A*"]["States"] = final[1]

INFO:root:Found a solution in 13 steps; visited 285 states


### A* (Manhattan Distance)

In [24]:
parent_state = dict()
state_cost = dict()

def h(state: State) -> int:
    r, c = np.argwhere(state.data == TARGET).min(axis=0)   # top-left cell of the 2x2
    return abs(r - GOAL_ROWS.start) + abs(c - GOAL_COLS.start)

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: state_cost[s] + h(s),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)

level2_dict_algo["A* (Manhattan)"]["Steps"] = len(final[0])
level2_dict_algo["A* (Manhattan)"]["States"] = final[1]

INFO:root:Found a solution in 13 steps; visited 227 states


## Level 4 - 5x4 board with 2x2 target, 4 2x1, 1 1x2, 4 1x1, 2 empty cells

In [25]:
level4_dict = results_scramble["Level 4"]
start = np.array([[1, -1, -1, 2],
                  [5, 3, 4, 6],
                  [5, 9, 9, 6],
                  [7, 0, 0, 8],
                  [7, 0, 0, 8]])
GOAL = State(start, shape_codes(start, TARGET))
GOAL_ROWS, GOAL_COLS = slice(3, 5), slice(1, 3)   # rows 3-4, columns 1-2
level4_dict["Goal"] = GOAL
logging.info(f"Goal: \n{GOAL}")

INFO:root:Goal: 
[[ 1 -1 -1  2]
 [ 5  3  4  6]
 [ 5  9  9  6]
 [ 7  0  0  8]
 [ 7  0  0  8]]


In [26]:
INITIAL_STATE = GOAL
for r in range(100_000):
    INITIAL_STATE = result(INITIAL_STATE, choice(list(double_loop(INITIAL_STATE))))

level4_dict["Initial State"] = INITIAL_STATE
logging.info(f"Initial State: \n{INITIAL_STATE}")

INFO:root:Initial State: 
[[ 8  9  9 -1]
 [ 8  0  0  2]
 [ 5  0  0  3]
 [ 5  4  7  6]
 [ 1 -1  7  6]]


In [27]:
level4_dict_algo = level4_dict["Algorithms"]

### Breadth - First

In [28]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)
level4_dict_algo["Breadth-First"]["Steps"] = len(final[0])
level4_dict_algo["Breadth-First"]["States"] = final[1]

INFO:root:Found a solution in 27 steps; visited 560 states


### Depth-First

In [29]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)
level4_dict_algo["Breadth-First"]["Steps"] = len(final[0])
level4_dict_algo["Breadth-First"]["States"] = final[1]

INFO:root:Found a solution in 27 steps; visited 560 states


### Greedy Best-First

In [30]:
parent_state = dict()
state_cost = dict()

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: -len(state_cost),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)
level4_dict_algo["Greedy Best-First"]["Steps"] = len(final[0])
level4_dict_algo["Greedy Best-First"]["States"] = final[1]

INFO:root:Found a solution in 319 steps; visited 748 states


A*

In [31]:
parent_state = dict()
state_cost = dict()


def h(state):
    return np.sum((state._data != GOAL._data) & (state._data > 0))

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: state_cost[s] + h(s),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)

level4_dict_algo["A*"]["Steps"] = len(final[0])
level4_dict_algo["A*"]["States"] = final[1]

INFO:root:Found a solution in 27 steps; visited 467 states


### A* (Manhattan Distance)

In [32]:
parent_state = dict()
state_cost = dict()

def h(state: State) -> int:
    r, c = np.argwhere(state.data == TARGET).min(axis=0)   # top-left cell of the 2x2
    return abs(r - GOAL_ROWS.start) + abs(c - GOAL_COLS.start)

final = search(
    INITIAL_STATE,
    goal_test=goal_test,
    parent_state=parent_state,
    state_cost=state_cost,
    priority_function=lambda s: state_cost[s] + h(s),
    unit_cost=lambda a: 1,
    goal_rows = GOAL_ROWS,
    goal_cols = GOAL_COLS,
)

level4_dict_algo["A* (Manhattan)"]["Steps"] = len(final[0])
level4_dict_algo["A* (Manhattan)"]["States"] = final[1]

INFO:root:Found a solution in 27 steps; visited 352 states
